# Setup

In [3]:
!pip install -q GPUtil
!pip install -q torchtext

In [ ]:
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
from torch.nn.parallel import DistributedDataParallel as DDP

from torch.optim.lr_scheduler import LambdaLR

import torch.distributed as dist
import torch.multiprocessing as mp

from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler

from torchtext.data.functional import to_map_style_dataset
from torchtext.vocab import build_vocab_from_iterator
import torchtext.datasets as datasets

import os, math, copy, time, spacy, GPUtil, warnings
import pandas as pd
import altair as alt


# Set to False to skip notebook execution (e.g. for debugging)
warnings.filterwarnings("ignore")
RUN_EXAMPLES = True

print(torch.__version__)
print("GPU Available: ", torch.cuda.is_available())

# Model Architecture

In [5]:
class EncoderDecoder(nn.Module):
    """
    A standard Encoder-Decoder architecture. Base for this and many other models.
    """

    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()
        # Step 1: Source Tokens -> Vectors (Embedding + Positional Encoding)
        self.src_embed = src_embed

        # Step 2: Take (src_embed Vectors + src_mask) -> (Self-Attention & FFN) -> Produce (Memory)
        self.encoder = encoder

        # Step 3: Target Tokens -> Vectors (Embedding + Positional Encoding)
        self.tgt_embed = tgt_embed

        # Step 4: Take (tgt_embed Vectors + Memory + Masks) -> Produce (Decoder Hidden Representation)
        self.decoder = decoder

        # Step 5: Take (Decoder Output) -> (Linear + LogSoftmax) -> Produce Probabilities [Vocab Size]
        self.generator = generator

    def forward(self, src, tgt, src_mask, tgt_mask):
        "Take in and process masked src and target sequences."
        # Step A: Encode source sequence into memory
        memory = self.encode(src, src_mask)
        # Step B: Decode target sequence using memory
        return self.decode(memory, src_mask, tgt, tgt_mask)

    def encode(self, src, src_mask):
        # Pass src through embedding first, then into encoder
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        # Pass tgt through embedding first, then into decoder with memory and masks
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

In [6]:
class Generator(nn.Module):
    "Define standard linear + softmax generation step."

    def __init__(self, d_model, vocab_size):
        super(Generator, self).__init__()
        # Initialize weight matrix to project from [d_model] -> [vocab_size]
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # 1. self.proj(x): Transform shape [Batch, Seq_Len, d_model] -> [Batch, Seq_Len, vocab_size] (Logits)
        # 2. log_softmax(..., dim=-1): Convert logits across the last dimension into log probabilities
        log_probs = log_softmax(self.proj(x), dim=-1)
        return log_probs

# Encoder and Decoder Stacks